# EE473 Final Project: Energy-Aware Edge Scheduling with Reinforcement Learning

This notebook presents the full results of a custom RL scheduling simulator built on real Google cluster workload traces.
A single edge device selects one of three discrete performance modes (low / medium / high) at each timestep to trade off energy consumption against queuing delay and deadline misses.
All figures and tables are regenerated from saved experiment artifacts.

## 1) Problem Setup: Custom Scheduling Environment and Reward Function

### Scheduling Scenario

This project addresses a **single-device edge scheduling** problem: one device faces a time-varying workload (from real Google cluster traces) and must select a performance mode at each timestep to trade off energy consumption against service quality.

### Custom Simulator Design

| Component | Definition |
|---|---|
| **State** | `(workload_bin, queue_bin, battery_bin)` — each dimension discretized into 5 bins |
| **Action** | 3 discrete performance modes: `low / medium / high` |
| **Transition** | Read current trace workload → service tasks under chosen mode → update queue backlog and battery |
| **Termination** | Episode length reached (288 steps) or battery depleted (optional hard cutoff) |

### Energy Model and Scheduling Constraints

Service rates and energy costs per mode (all constants centralized in `src/config.py`):

| Mode | Service Rate (units/step) | Energy Cost (units/step) |
|---|---|---|
| low | 0.35 | 0.18 |
| medium | 0.75 | 0.42 |
| high | 1.20 | 0.85 |

Scheduling constraints:
- Queue cap: `max_queue = 8.0` (clipped above this)
- Deadline miss threshold: `deadline_queue_threshold = 2.5` (queue exceeding this counts as a miss)
- Mode-switch penalty: `switch_cost = 0.05` (extra energy when switching modes)
- Battery capacity: `battery_capacity = 120.0` (hard cutoff disabled in main experiments)

### Reward Function

$$r_t = -(\alpha \cdot E_t + \beta \cdot L_t + \gamma \cdot M_t)$$

Where:
- $E_t$: energy consumed this step (including switch penalty)
- $L_t$: queue length (proxy for queuing delay)
- $M_t$: deadline miss indicator (0/1)
- Default weights: $\alpha=1.0,\; \beta=0.6,\; \gamma=2.0$

**Design rationale**: The reward jointly penalizes energy, queuing delay, and deadline violations with tunable weights, encouraging the agent to dynamically adapt its mode rather than committing to a fixed strategy.

## 2) Dataset and Experiment Protocol

### Dataset

- **Source**: Google cluster workload trace (task events, 2011 public release)
- **Shards used**: 120 compressed CSV shards (`task_events_part_00000` – `part_00119`)
- **Preprocessing**: submit-event counts aggregated into 60-second bins, normalized to [0, 1], then segmented into fixed-length episodes
- **Episode length**: 288 steps (≈ 4.8 hours of real cluster time per episode)

### Train / Test Split

| Split | Episodes per seed |
|---|---|
| Training | 20 episodes |
| Test (non-overlapping) | 6 episodes |

### RL Methods Compared

| Method | Description |
|---|---|
| **Tabular Q-learning** | Q-table over discrete state space (125 states × 3 actions); ε-greedy exploration |
| **Linear Approx Q-learning** | Linear function approximation with hand-crafted feature vector; same ε-greedy schedule |

### Baselines

`always_low`, `always_medium`, `always_high`, and a threshold heuristic (switch to high mode when workload exceeds threshold `w`).

### Evaluation Protocol

- **5 random seeds** per method
- Report **mean ± std** for return, energy, latency, miss rate, and training wall time
- Ablations: reward weight sensitivity, hyperparameter sensitivity, train/test slicing generalization

## 3) Environment Walkthrough: Loading Data and Running a Single Episode

This section imports the project modules, loads the workload trace, and runs a complete episode with a random policy to demonstrate the simulator interface step by step.

In [ ]:
from pathlib import Path
import subprocess
import json

ROOT = Path.cwd().resolve().parent
assert (ROOT / 'scripts').exists(), f'Expected repo root at {ROOT}'
print('ROOT =', ROOT)

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Add src/ to path so we can import project modules directly
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from config import DEFAULT_ENV_CONFIG
from data_loader import load_workload_trace, build_episodes
from env import EnergySchedulingEnv

# ── Load workload trace ──────────────────────────────────────────────────────
TRACE_PATH = ROOT / 'data/processed/workload_trace_120.csv'
EPISODE_LENGTH = 288
STRIDE = 288

train_series = load_workload_trace(TRACE_PATH, split='train')
test_series  = load_workload_trace(TRACE_PATH, split='test')

train_episodes = build_episodes(train_series, EPISODE_LENGTH, stride=STRIDE)[:20]
test_episodes  = build_episodes(test_series,  EPISODE_LENGTH, stride=STRIDE)[:6]

print(f"Train episodes : {len(train_episodes)}  ×  {len(train_episodes[0])} steps")
print(f"Test  episodes : {len(test_episodes)}  ×  {len(test_episodes[0])} steps")
print(f"Workload range : [{min(train_series):.3f}, {max(train_series):.3f}]")

In [ ]:
# ── Visualise the first training episode's workload ──────────────────────────
fig, ax = plt.subplots(figsize=(10, 2.5))
ax.plot(train_episodes[0], linewidth=0.9, color='steelblue')
ax.set_xlabel('Timestep (1 step = 60 s)')
ax.set_ylabel('Normalised workload')
ax.set_title('Workload trace — first training episode (288 steps ≈ 4.8 h)')
ax.set_xlim(0, EPISODE_LENGTH - 1)
plt.tight_layout()
plt.show()

In [ ]:
# ── Run one full episode with a random policy; print first 5 steps ───────────
rng = np.random.default_rng(42)
env = EnergySchedulingEnv(train_episodes[0], config=DEFAULT_ENV_CONFIG)
obs, info = env.reset()

print(f"{'Step':>4}  {'Action':>6}  {'Workload':>8}  {'Queue':>6}  "
      f"{'Battery':>7}  {'Energy':>6}  {'Latency':>7}  {'Miss':>4}  {'Reward':>8}")
print('-' * 72)

step_log = []
done = False
t = 0
while not done:
    action = int(rng.integers(0, 3))
    obs, reward, done, info = env.step(action)
    step_log.append({
        'workload': info['arrival_norm'],
        'queue':    info['queue'],
        'action':   action,
        'energy':   info['step_energy'],
        'reward':   info['step_reward'],
    })
    if t < 5:
        print(f"{t+1:>4}  {DEFAULT_ENV_CONFIG.action_names[action]:>6}  "
              f"{info['arrival_norm']:>8.3f}  {info['queue']:>6.3f}  "
              f"{info['battery']:>7.2f}  {info['step_energy']:>6.3f}  "
              f"{info['step_latency']:>7.3f}  {int(info['step_miss']):>4}  "
              f"{info['step_reward']:>8.4f}")
    t += 1

print(f"\n  ... (episode length = {t} steps)")
total_return = sum(s['reward'] for s in step_log)
print(f"  Total episode return (random policy): {total_return:.3f}")

## 4) Baseline Policies: Evaluation on Test Episodes

We evaluate four heuristic policies on the same test episodes to establish a non-learning performance baseline.

In [ ]:
from baselines import (
    always_low_policy, always_medium_policy, always_high_policy,
    threshold_policy_factory, evaluate_policy,
)

baseline_policies = {
    'always_low':    always_low_policy,
    'always_medium': always_medium_policy,
    'always_high':   always_high_policy,
    'threshold(w=0.60)': threshold_policy_factory(workload_threshold=0.60),
    'threshold(w=0.80)': threshold_policy_factory(workload_threshold=0.80),
}

print(f"{'Policy':<22}  {'Avg Return':>10}  {'Energy':>7}  {'Latency':>8}  {'Miss Rate':>9}")
print('-' * 65)
baseline_results = {}
for name, policy in baseline_policies.items():
    m = evaluate_policy(test_episodes, policy, config=DEFAULT_ENV_CONFIG)
    baseline_results[name] = m
    print(f"{name:<22}  {m['avg_episode_return']:>10.3f}  "
          f"{m['avg_step_energy']:>7.4f}  {m['avg_step_latency']:>8.4f}  "
          f"{m['miss_rate']:>9.4f}")

best_baseline_name   = max(baseline_results, key=lambda k: baseline_results[k]['avg_episode_return'])
best_baseline_return = baseline_results[best_baseline_name]['avg_episode_return']
print(f"\n  → Best baseline: {best_baseline_name}  (avg return = {best_baseline_return:.3f})"
      f"\n    RL methods in Section 5 will be compared against this value.")

## 5) RL Training: Tabular Q-Learning and Linear Approximation Q-Learning

We train both RL methods from scratch (single seed, full epoch count matching the main experiment) and report learning curves and final test metrics.
For the complete multi-seed results, see Section 8.

In [ ]:
import time
from q_learning import QLearningConfig, train_tabular_q_learning

# ── Tabular Q-learning (300 epochs, single seed) ─────────────────────────────
tabular_cfg = QLearningConfig(
    num_epochs=300,
    alpha=0.15,
    gamma=0.98,
    epsilon_start=0.30,
    epsilon_end=0.02,
    epsilon_decay=0.995,
    eval_every=5,
    seed=42,
)

t0 = time.perf_counter()
tabular_result = train_tabular_q_learning(
    train_episodes=train_episodes,
    test_episodes=test_episodes,
    env_config=DEFAULT_ENV_CONFIG,
    algo_config=tabular_cfg,
)
tabular_time = time.perf_counter() - t0

print(f"Tabular Q-learning  —  training time: {tabular_time:.2f} s")
bm = tabular_result['best_test_metrics']
print(f"  Best checkpoint (epoch {int(tabular_result['best_epoch'])}):")
print(f"    avg_return = {bm['avg_episode_return']:.4f}")
print(f"    energy     = {bm['avg_step_energy']:.4f}")
print(f"    latency    = {bm['avg_step_latency']:.4f}")
print(f"    miss_rate  = {bm['miss_rate']:.4f}")

In [ ]:
from approx_q import ApproxQLearningConfig, train_linear_approx_q_learning

# ── Linear Approximation Q-learning (400 epochs, single seed) ────────────────
approx_cfg = ApproxQLearningConfig(
    num_epochs=400,
    alpha=0.02,
    gamma=0.98,
    epsilon_start=0.30,
    epsilon_end=0.02,
    epsilon_decay=0.997,
    eval_every=5,
    seed=42,
)

t0 = time.perf_counter()
approx_result = train_linear_approx_q_learning(
    train_episodes=train_episodes,
    test_episodes=test_episodes,
    env_config=DEFAULT_ENV_CONFIG,
    algo_config=approx_cfg,
)
approx_time = time.perf_counter() - t0

print(f"Linear Approx Q-learning  —  training time: {approx_time:.2f} s")
print(f"  Feature vector dimension: {int(approx_result['feature_dim'])}")
bm = approx_result['best_test_metrics']
print(f"  Best checkpoint (epoch {int(approx_result['best_epoch'])}):")
print(f"    avg_return = {bm['avg_episode_return']:.4f}")
print(f"    energy     = {bm['avg_step_energy']:.4f}")
print(f"    latency    = {bm['avg_step_latency']:.4f}")
print(f"    miss_rate  = {bm['miss_rate']:.4f}")

In [ ]:
# ── Plot learning curves for both methods ────────────────────────────────────
tab_hist  = tabular_result['history']
apx_hist  = approx_result['history']

tab_epochs  = [h['epoch']               for h in tab_hist]
tab_returns = [h['eval_test_avg_return'] for h in tab_hist]
apx_epochs  = [h['epoch']               for h in apx_hist]
apx_returns = [h['eval_test_avg_return'] for h in apx_hist]

# best baseline for reference line
best_baseline_return = max(v['avg_episode_return'] for v in baseline_results.values())

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(tab_epochs, tab_returns, label='Tabular Q-learning (300 epochs)',       color='steelblue',  linewidth=1.5)
ax.plot(apx_epochs, apx_returns, label='Linear Approx Q-learning (400 epochs)', color='darkorange', linewidth=1.5)
ax.axhline(best_baseline_return, linestyle='--', color='gray', linewidth=1.2,
           label=f'Best baseline ({best_baseline_return:.2f})')
ax.set_xlabel('Epoch')
ax.set_ylabel('Avg test return')
ax.set_title('Learning curves — full training run (seed=42)')
ax.legend()
plt.tight_layout()
plt.show()

print(f"\nTraining time  |  Tabular: {tabular_time:.1f} s   Approx: {approx_time:.1f} s")

# ── Single-seed comparison vs best baseline ───────────────────────────────────
tab_best = tabular_result['best_test_metrics']['avg_episode_return']
apx_best = approx_result['best_test_metrics']['avg_episode_return']

print(f"\n{'─'*55}")
print(f"  Single-seed result vs best baseline")
print(f"{'─'*55}")
print(f"  {'Method':<32}  {'Avg Return':>10}  {'Δ vs baseline':>13}")
print(f"  {'─'*32}  {'─'*10}  {'─'*13}")
best_bl_label = max(baseline_results, key=lambda k: baseline_results[k]['avg_episode_return'])
print(f"  {f'Best baseline ({best_bl_label})':<32}  {best_baseline_return:>10.4f}  {'(reference)':>13}")
print(f"  {'Tabular Q-learning':<32}  {tab_best:>10.4f}  {tab_best - best_baseline_return:>+13.4f}")
print(f"  {'Linear Approx Q-learning':<32}  {apx_best:>10.4f}  {apx_best - best_baseline_return:>+13.4f}")
print(f"{'─'*55}")

## 6) Saved Best Models: Loading and Inspecting Learned Policies

We reload the best Q-table and weight vector from the full 5-seed experiment (stored in `results/`) and evaluate them on the test set.
This section also documents the feature vector design for the linear approximation method.

In [ ]:
from q_learning import evaluate_q_policy, greedy_action as tab_greedy
from approx_q import LinearQApproximator, evaluate_linear_q_policy

# ── Load pre-trained artifacts from full experiment ───────────────────────────
TAB_Q_PATH  = ROOT / 'results/tabular_q_learning_120/q_table_best.npy'
APX_W_PATH  = ROOT / 'results/approx_q_learning_120/weights_best.npy'

q_table_best  = np.load(TAB_Q_PATH)
weights_best  = np.load(APX_W_PATH)
approximator  = LinearQApproximator(DEFAULT_ENV_CONFIG)

print("Q-table shape :", q_table_best.shape,
      "  (workload_bins × queue_bins × battery_bins × actions)")
print("Weight vector  :", weights_best.shape,
      f"  ({approximator.base_dim} base features × {approximator.num_actions} actions)")

# ── Evaluate saved models on test episodes ────────────────────────────────────
tab_saved = evaluate_q_policy(q_table_best, test_episodes, env_config=DEFAULT_ENV_CONFIG)
apx_saved = evaluate_linear_q_policy(weights_best, approximator, test_episodes,
                                      env_config=DEFAULT_ENV_CONFIG)

print("\n  Saved best model evaluation on test set:")
print(f"  {'Method':<28}  {'Avg Return':>10}  {'Energy':>7}  {'Latency':>8}  {'Miss Rate':>9}")
print('  ' + '-' * 65)
for label, m in [('Tabular Q (saved best)', tab_saved), ('Approx Q (saved best)', apx_saved)]:
    print(f"  {label:<28}  {m['avg_episode_return']:>10.4f}  "
          f"{m['avg_step_energy']:>7.4f}  {m['avg_step_latency']:>8.4f}  "
          f"{m['miss_rate']:>9.4f}")

### Feature Vector Design for Linear Approximation Q-Learning

The linear approximation represents Q(s, a) = **w**ₐᵀ **φ**(s) using a block-structured weight vector.
Each action has its own dedicated block; the base feature vector **φ**(s) is:

| Index | Feature | Description |
|---|---|---|
| 0 | 1.0 | Bias term |
| 1 | workload (continuous) | Normalised workload arrival at current step |
| 2 | queue_norm | Queue length normalised by max_queue |
| 3 | battery_ratio | Remaining battery / capacity |
| 4–8 | one-hot workload bin | Which of 5 workload bins the state falls in |
| 9–13 | one-hot queue bin | Which of 5 queue bins |
| 14–18 | one-hot battery bin | Which of 5 battery bins |
| 19 | workload × queue_norm | Interaction: high load + high queue → need high mode |
| 20 | queue_norm × (1 − battery) | Interaction: high queue + low battery → penalise high mode |
| 21 | workload × (1 − battery) | Interaction: high load + low battery → constrained choice |

**Total**: 22 base features × 3 actions = **66-dimensional weight vector**.

The one-hot bins preserve tabular-like locality while the continuous and interaction terms allow the model to generalise across bin boundaries — giving linear approximation an advantage over a pure Q-table on partially visited states.

In [ ]:
# ── Visualise learned Q-table: greedy action heatmap across battery bins ─────
action_names = DEFAULT_ENV_CONFIG.action_names
n_w   = len(DEFAULT_ENV_CONFIG.workload_bins) - 1   # 5
n_q   = len(DEFAULT_ENV_CONFIG.queue_bins)    - 1   # 5
n_b   = len(DEFAULT_ENV_CONFIG.battery_bins)  - 1   # 5

fig, axes = plt.subplots(1, n_b, figsize=(13, 3), sharey=True)
fig.suptitle(
    'Tabular Q-table: greedy action by (workload_bin, queue_bin) for each battery_bin\n'
    '(0 = low battery → 4 = full battery)',
    y=1.04,
)

cmap = plt.cm.get_cmap('viridis', 3)
for b_bin in range(n_b):
    ax = axes[b_bin]
    grid = np.zeros((n_q, n_w), dtype=int)
    for w in range(n_w):
        for q in range(n_q):
            grid[q, w] = int(np.argmax(q_table_best[w, q, b_bin]))   # ← use b_bin
    im = ax.imshow(grid, vmin=0, vmax=2, cmap=cmap, aspect='auto', origin='lower')
    ax.set_title(f'batt_bin={b_bin}')
    ax.set_xlabel('workload_bin')
    if b_bin == 0:
        ax.set_ylabel('queue_bin')
    ax.set_xticks(range(n_w))
    ax.set_yticks(range(n_q))

cbar = fig.colorbar(im, ax=axes, ticks=[0, 1, 2], shrink=0.85)
cbar.ax.set_yticklabels(action_names)
plt.tight_layout()
plt.show()

## 7) Regenerate Saved Experiment Artifacts

This cell rebuilds all publication-style figures and tables from pre-computed results into `results/final_figures/`.
Skip this cell if artifacts already exist.

In [ ]:
cmd = [
    'python3', str(ROOT / 'scripts/generate_final_artifacts.py'),
    '--trace-path', str(ROOT / 'data/processed/workload_trace_120.csv'),
    '--phase3-json', str(ROOT / 'results/phase3_multiseed_120.json'),
    '--reward-json', str(ROOT / 'results/reward_sensitivity_120.json'),
    '--hyper-json', str(ROOT / 'results/hyperparam_sensitivity_120.json'),
    '--generalization-json', str(ROOT / 'results/generalization_check_120.json'),
    '--tabular-history-csv', str(ROOT / 'results/tabular_q_learning_120/history.csv'),
    '--approx-history-csv', str(ROOT / 'results/approx_q_learning_120/history.csv'),
    '--tabular-q-table', str(ROOT / 'results/tabular_q_learning_120/q_table_best.npy'),
    '--approx-weights', str(ROOT / 'results/approx_q_learning_120/weights_best.npy'),
    '--output-dir', str(ROOT / 'results/final_figures'),
]
res = subprocess.run(cmd, capture_output=True, text=True, check=True)
print(res.stdout)

## 8) Full Experiment Results: Summary Table and Key Figures

In [ ]:
from IPython.display import Markdown, display as ipy_display

summary_md_path = ROOT / 'results/final_figures/final_summary_table.md'
ipy_display(Markdown(summary_md_path.read_text(encoding='utf-8')))

### Figures

In [ ]:
from IPython.display import Image, display

fig_dir = ROOT / 'results/final_figures'
figure_names = [
    'learning_curves_tabular_vs_approx.png',
    'baseline_vs_rl_return.png',
    'energy_latency_tradeoff.png',
    'reward_sensitivity_return.png',
    'hyperparam_sensitivity_return.png',
    'policy_action_frequency.png',
]
for name in figure_names:
    path = fig_dir / name
    print('\n', name)
    display(Image(filename=str(path)))

## 9) Key Parameter Analysis

### Effect of Reward Weights α/β/γ

Ablation results from `results/reward_sensitivity_120.md` (5 seeds, approx Q-learning):

| (α, β, γ) | Avg Return | Energy | Latency |
|---|---|---|---|
| (0.80, 0.60, 2.00) | **-45.10 ± 0.05** | 0.190 | 0.008 |
| (1.00, 0.40, 2.00) | -55.40 ± 0.11 | 0.184 | 0.021 |
| **(1.00, 0.60, 2.00)** ← default | -56.03 ± 0.11 | 0.190 | 0.008 |
| (1.00, 0.80, 2.00) | -56.46 ± 0.11 | 0.190 | 0.007 |
| (1.20, 0.60, 2.00) | -66.00 ± 0.13 | 0.190 | 0.007 |

**Findings**:
- **α (energy weight)** has the largest effect: reducing α from 1.0 to 0.8 improves return by ~11 points — the agent is willing to use high mode more freely when energy is penalized less. Increasing α to 1.2 suppresses high-mode use and hurts return.
- **β (latency weight)** has minor impact: across 0.4–0.8, return changes by less than 1 point.
- **γ (miss penalty)** shows no practical effect because miss rate is 0 in all configurations — the deadline constraint is easily satisfied under this workload.

### Effect of Algorithm Hyperparameters

Results from `results/hyperparam_sensitivity_120.md` (learning rate α_lr, ε-decay, discount γ_rl):

| (α_lr, ε-decay, γ_rl) | Avg Return | std |
|---|---|---|
| **(0.030, 0.999, 0.980)** ← best | -56.017 | 0.144 |
| (0.020, 0.997, 0.980) | -56.027 | 0.111 |
| (0.020, 0.997, 0.950) | -56.063 | 0.141 |
| (0.015, 0.995, 0.980) | -56.217 | 0.314 |
| (0.020, 0.997, 0.990) | -56.267 | 0.265 |

**Findings**:
- Return differences across all hyperparameter settings are very small (max gap ~0.25), indicating the algorithm is robust to hyperparameter choice within a reasonable range.
- Smaller learning rate (0.015) and higher discount (0.990) roughly double the variance across seeds, suggesting slightly slower or less stable convergence.
- ε-decay has negligible effect on return, indicating that exploration is not a bottleneck in this environment.

### Training Time Comparison

| Method | Training Time (5-seed mean) |
|---|---|
| Tabular Q-learning | **6.3 ± 0.04 s** |
| Linear Approx Q-learning | 34.4 ± 0.17 s |

Tabular Q-learning is approximately **5.4× faster** than linear approximation. The main reason is that tabular lookup is O(1) per step while linear approximation requires a dot product over the feature vector. The return gap between the two methods is only ~0.18 points (-56.20 vs -56.02), making tabular Q-learning more cost-effective when training time is limited.

## 10) Expected vs. Actual Results

### What We Expected

Before running experiments, we had the following theoretical expectations:

1. **RL should outperform heuristic baselines** — a learned policy can adapt to workload patterns over an episode, whereas a fixed-threshold heuristic cannot.
2. **Linear approximation should outperform tabular Q-learning** — the feature representation generalizes value estimates across unseen state combinations, which matters when the state space (5×5×5 = 125 states) is only partially visited during training.
3. **Tabular Q-learning should train faster** — direct table lookup avoids per-step dot products required by function approximation.
4. **Results should be stable across seeds** — the environment dynamics are deterministic given the trace and seed, so variance should be low.

### What We Actually Observed

| Expectation | Observed Outcome | Match? |
|---|---|---|
| RL beats baselines | Approx Q: -56.02, Tabular Q: -56.20 vs. best baseline -57.29 | ✅ Yes, but margin is small (~1.3 pts) |
| Approx Q better than Tabular Q | -56.02 vs. -56.20 (approx wins by 0.18 pts) | ✅ Yes, but difference is marginal |
| Tabular trains faster | 6.3s vs. 34.4s | ✅ Yes, ~5.4× speedup confirmed |
| Low variance across seeds | RL std ≤ 0.12, baseline std = 0.00 | ✅ Yes, results are highly reproducible |

### Why the RL Margin is Small

The narrow improvement (~1.3 points over best baseline) is expected given the environment structure:
- The **always-low** policy achieves near-optimal energy efficiency because the Google cluster trace has many low-workload intervals; a simple strategy already handles most timesteps well.
- The state space (125 states) is relatively small and well-covered by heuristics, leaving limited room for learned policies to exploit structure.
- All methods achieve **zero deadline miss rate** — the problem is not challenging enough on that dimension to differentiate learned vs. heuristic behavior.

This is consistent with the intended **narrow scope** of the project: the goal was a clearly defined, reproducible RL formulation rather than a difficult benchmark where deep RL would be necessary.

## 11) Challenges and Future Work

### Challenges Encountered

**1. Environment design — discretization trade-offs**
Choosing the number and boundaries of state bins required balancing expressiveness against the tabular method's coverage requirement. Too many bins leave Q-values uninitialized; too few bins collapse important state distinctions. We settled on 5 bins per dimension (125 total states) after verifying that training episodes provide sufficient state coverage.

**2. Reward function calibration**
The initial reward formulation had α=β=γ=1.0, which caused the agent to over-prioritize miss avoidance and select `high` mode almost always. Tuning the weights (final: α=1.0, β=0.6, γ=2.0) was needed to produce a policy that meaningfully varies its mode selection across workload levels.

**3. Test episode coverage**
The original 20-shard dataset produced only 2–3 non-overlapping test episodes, making evaluation noisy. Expanding to 120 shards yielded 6 non-overlapping test episodes per seed — sufficient for stable mean/std reporting.

**4. Marginal RL advantage**
Because the `always-low` baseline already performs well on this trace (low-workload periods dominate), the RL advantage over the best baseline is modest (~1.3 points). Demonstrating a clear learning advantage required careful baseline selection and multi-seed averaging.

### Future Work

- **Multi-device coordination**: extend from a single device to a cluster with shared workload queues and resource contention, where RL has a stronger advantage over simple heuristics.
- **Richer constraints**: add mode-switch calibration overhead, hard service-level agreement targets, and non-linear battery discharge curves for more realistic energy modeling.
- **Larger action space or continuous control**: explore finer-grained frequency scaling or actor-critic methods (e.g., PPO) to better exploit continuous workload patterns.
- **Workload diversity**: evaluate across additional trace windows and domain-shifted workloads (e.g., bursty or periodic patterns) to test generalization beyond a single trace family.
- **Risk-sensitive objectives**: incorporate tail-latency penalties (e.g., CVaR) or uncertainty-aware reward shaping for safety-critical scheduling scenarios.

## 12) Conclusion

### Main Results (5 seeds, 120-shard dataset, non-overlapping test episodes)

| Method | Avg Return (mean ± std) | Energy | Latency | Miss Rate | Train Time (s) |
|---|---|---|---|---|---|
| Linear Approx Q-learning | **-56.016 ± 0.117** | 0.190 ± 0.001 | 0.008 ± 0.001 | 0.000 | 34.4 ± 0.17 |
| Tabular Q-learning | -56.200 ± 0.084 | 0.191 ± 0.001 | 0.007 ± 0.001 | 0.000 | **6.3 ± 0.04** |
| Baseline: always_low | -57.289 ± 0.000 | 0.180 ± 0.000 | 0.032 ± 0.000 | 0.000 | — |

### Key Takeaways

- **Both RL methods outperform the best heuristic baseline** in average return on held-out test episodes, confirming that the learned policies extract value from workload patterns that fixed rules cannot.
- **Linear approximation achieves slightly higher return** (-56.02 vs. -56.20) because its feature representation generalizes across the 125-state space more effectively than tabular lookup under limited training data.
- **Tabular Q-learning is 5.4× faster to train** (6.3s vs. 34.4s) with only marginal return loss — a favorable trade-off when training time is constrained.
- **All methods achieve zero deadline miss rate**, indicating the workload is manageable even with conservative policies; the main differentiation is in energy-latency balance.
- **Results are stable across 5 random seeds** (std ≤ 0.12 for RL, 0.00 for deterministic baselines), validating the reproducibility of the evaluation protocol.
- **Generalization across slicing settings is consistent**: non-overlap vs. overlap test slicing differs by only ~0.36 points, showing the policy transfers well across episode boundaries.